# 02. Проверка эксперимента
Размеры групп, баланс групп, общий эффект SMS.

In [6]:
import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

DATA = 'data/data/'
train = pd.read_csv(DATA + 'uplift_train.csv')
clients = pd.read_csv(DATA + 'clients.csv', parse_dates=['first_issue_date', 'first_redeem_date'])
df = train.merge(clients, on='client_id', how='left')

print(df.shape, '| client_id уникален:', df['client_id'].is_unique, '| без данных клиента:', df['age'].isna().sum())

(200039, 7) | client_id уникален: True | без данных клиента: 0


## Размеры групп (SRM)
Проверка, что группы разделились 50/50 без перекоса.

In [7]:
sizes = df['treatment_flg'].value_counts().sort_index()
print(sizes)
print(chisquare(sizes))

treatment_flg
0    100058
1     99981
Name: count, dtype: int64
Power_divergenceResult(statistic=np.float64(0.029639220352031353), pvalue=np.float64(0.8633113903439603))


## Баланс групп
Группы должны быть похожи по возрасту, полу, стажу карты и использованию баллов.

In [8]:
df['card_days'] = (pd.Timestamp('2019-03-18') - df['first_issue_date']).dt.days
df['redeemed'] = df['first_redeem_date'].notna()
age_ok = df['age'].between(14, 100)  # в данных есть невозможные возрасты

print('доля корректных возрастов:', round(age_ok.mean(), 4))
print(df[age_ok].groupby('treatment_flg')['age'].median())
print(df.groupby('treatment_flg')[['card_days', 'redeemed']].mean())
print(pd.crosstab(df['treatment_flg'], df['gender'], normalize='index').round(3))

доля корректных возрастов: 0.9956
treatment_flg
0    45.0
1    45.0
Name: age, dtype: float64
                card_days  redeemed
treatment_flg                      
0              431.359961  0.908973
1              433.101099  0.915604
gender             F      M      U
treatment_flg                     
0              0.368  0.168  0.465
1              0.369  0.167  0.464


## Общий эффект SMS
Конверсия в покупку с SMS и без, разница и 95% доверительный интервал.

In [9]:
conv = df.groupby('treatment_flg')['target'].agg(['sum', 'count', 'mean'])
print(conv)

buy = conv.loc[[1, 0], 'sum'].values
n = conv.loc[[1, 0], 'count'].values
z, p = proportions_ztest(buy, n)
low, high = confint_proportions_2indep(buy[0], n[0], buy[1], n[1])

diff = conv.loc[1, 'mean'] - conv.loc[0, 'mean']
print(f'эффект: {diff*100:+.2f} п.п., 95% ДИ [{low*100:+.2f}; {high*100:+.2f}], p = {p:.2g}')

                 sum   count      mean
treatment_flg                         
0              60363  100058  0.603280
1              63639   99981  0.636511
эффект: +3.32 п.п., 95% ДИ [+2.90; +3.75], p = 6.6e-53


In [10]:
print('последняя дата выдачи карты:', df['first_issue_date'].max())
print('последняя дата первого списания:', df['first_redeem_date'].max())
print('списаний после 18.03.2019:', (df['first_redeem_date'] > '2019-03-18').sum())

df['redeemed_before'] = df['first_redeem_date'] <= '2019-03-18'
print(df.groupby('treatment_flg')['redeemed_before'].mean())

последняя дата выдачи карты: 2019-03-15 21:44:14
последняя дата первого списания: 2019-11-20 01:14:10
списаний после 18.03.2019: 22785
treatment_flg
0    0.796978
1    0.799792
Name: redeemed_before, dtype: float64


**Итог проверки:**
- Размеры групп: 100 058 / 99 981, SRM нет (p ≈ 0,86).
- Возраст, пол, стаж карты сбалансированы.
- first_redeem_date содержит даты после рассылки (22 785 клиентов, до 20.11.2019): это следствие эксперимента, а не признак клиента. С отсечкой по 18.03.2019 баланс есть (79,7% / 80,0%). В сегментации используется только история до 18.03.2019.
- Общий эффект SMS: конверсия 60,33% → 63,65%, +3,32 п.п., 95% ДИ [+2,90; +3,75], p < 0,001.
- 60% клиентов покупают и без SMS: рассылка даёт около 5% покупок среди получивших её.